# Time-Series Forecasting

Rolling-origin backtesting, decomposition, and forecast evaluation.
Key rule: **never use a standard train/test split for time series** — it leaks
future information into training.

## Setup

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "figure.dpi":        130,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "axes.titlesize":    13,
    "axes.labelsize":    11,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "legend.fontsize":   10,
    "font.family":       "sans-serif",
    "lines.linewidth":   2.2,
    "patch.edgecolor":   "none",
})
C = ["#2563EB","#DC2626","#16A34A","#D97706","#7C3AED","#0891B2","#DB2777"]
print("Setup complete")

## 1. Decomposition — trend, seasonality, noise

Before modelling, decompose the series to understand its components.
Additive decomposition: `y = trend + seasonality + residual`.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

rng = np.random.default_rng(42)
t   = np.arange(144)

# Synthetic series: trend + monthly seasonality + noise
trend      = 100 + 0.8*t
seasonality = 20 * np.sin(2*np.pi*t/12)
noise       = rng.normal(0, 5, 144)
y           = trend + seasonality + noise
dates       = pd.date_range("2012-01", periods=144, freq="MS")
series      = pd.Series(y, index=dates)

result = seasonal_decompose(series, model="additive", period=12)

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
for ax, data, title, col in zip(axes,
    [series, result.trend, result.seasonal, result.resid],
    ["Original", "Trend", "Seasonal (period=12)", "Residual"],
    [C[0], C[2], C[3], C[1]]
):
    ax.plot(data.index, data.values, color=col, lw=1.8)
    ax.fill_between(data.index, data.values, alpha=0.15, color=col)
    ax.set_ylabel(title, fontsize=10)
    ax.grid(True, alpha=0.3)
axes[0].set_title("Additive decomposition: trend + seasonality + residual", fontsize=12)
plt.tight_layout()
plt.show()
print(f"Trend range: {result.trend.dropna().min():.0f} to {result.trend.dropna().max():.0f}")
print(f"Seasonal amplitude: +/-{result.seasonal.max():.1f}")
print(f"Residual std: {result.resid.dropna().std():.2f}  (noise level)")

## 2. Rolling-origin backtest

Split data into an expanding training window, forecast horizon steps ahead,
collect errors. This mirrors real deployment: you always forecast into the future.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA

def seasonal_naive(train, h, period=12):
    last_season = train.values[-period:]
    reps = (h // period) + 1
    return pd.Series(
        np.tile(last_season, reps)[:h],
        index=pd.date_range(train.index[-1], periods=h+1, freq="MS")[1:]
    )

def mape(actual, forecast):
    return np.mean(np.abs((actual - forecast) / actual)) * 100

# Rolling-origin evaluation
origin_step = 6
h           = 6
origins     = range(72, len(series) - h, origin_step)
results     = {"Seasonal Naive": [], "ETS": [], "ARIMA(1,1,1)x12": []}

for origin in origins:
    train  = series.iloc[:origin]
    actual = series.iloc[origin:origin+h]

    # Naive
    fc_naive = seasonal_naive(train, h)
    results["Seasonal Naive"].append(mape(actual.values, fc_naive.values))

    # ETS
    try:
        fc_ets = ExponentialSmoothing(train, trend="add", seasonal="add",
                                       seasonal_periods=12).fit(optimized=True).forecast(h)
        results["ETS"].append(mape(actual.values, fc_ets.values))
    except Exception:
        results["ETS"].append(np.nan)

    # ARIMA
    try:
        fc_arima = ARIMA(train, order=(1,1,1), seasonal_order=(1,1,1,12)).fit().forecast(h)
        results["ARIMA(1,1,1)x12"].append(mape(actual.values, fc_arima.values))
    except Exception:
        results["ARIMA(1,1,1)x12"].append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot of MAPE distributions
ax = axes[0]
data_box = [np.array(v)[~np.isnan(v)] for v in results.values()]
bp = ax.boxplot(data_box, labels=list(results.keys()), patch_artist=True,
                medianprops=dict(color="white", lw=2))
for patch, col in zip(bp["boxes"], C):
    patch.set_facecolor(col)
    patch.set_alpha(0.75)
ax.set(ylabel="MAPE (%)", title="Rolling-origin backtest
MAPE distribution by model")

# Mean MAPE per origin
ax = axes[1]
origin_idx = list(range(len(list(results.values())[0])))
for (name, vals), col in zip(results.items(), C):
    clean = np.array(vals, dtype=float)
    ax.plot(origin_idx, clean, "o-", color=col, alpha=0.8, ms=4, label=name)
ax.set(xlabel="Backtest origin index", ylabel="MAPE (%)",
       title="MAPE across rolling origins
(lower = better, stability matters too)")
ax.legend()

plt.tight_layout()
plt.show()
for name, vals in results.items():
    clean = [v for v in vals if not np.isnan(v)]
    print(f"{name:<22}  mean MAPE = {np.mean(clean):.1f}%  (n={len(clean)} origins)")

## 3. Forecast uncertainty — prediction intervals

A point forecast without intervals is incomplete. Prediction intervals
quantify how much the forecast could plausibly be wrong.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

train = series.iloc[:120]
test  = series.iloc[120:]
h     = len(test)

fit = ExponentialSmoothing(train, trend="add", seasonal="add",
                            seasonal_periods=12).fit(optimized=True)
fc  = fit.forecast(h)

# Bootstrap prediction intervals
n_boot = 500
boot_forecasts = np.zeros((n_boot, h))
resid = fit.resid.values
for b in range(n_boot):
    boot_resid = np.random.choice(resid, size=len(train), replace=True)
    boot_series = pd.Series(train.values + boot_resid, index=train.index)
    try:
        boot_fit = ExponentialSmoothing(boot_series, trend="add", seasonal="add",
                                         seasonal_periods=12).fit(optimized=True)
        boot_forecasts[b] = boot_fit.forecast(h).values
    except Exception:
        boot_forecasts[b] = fc.values

pi_80_lo = np.percentile(boot_forecasts, 10, axis=0)
pi_80_hi = np.percentile(boot_forecasts, 90, axis=0)
pi_95_lo = np.percentile(boot_forecasts, 2.5, axis=0)
pi_95_hi = np.percentile(boot_forecasts, 97.5, axis=0)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train.index[-36:], train.values[-36:], color=C[0], lw=2, label="Training (last 3 yrs)")
ax.plot(test.index, test.values, color="black", lw=2, label="Actual")
ax.plot(fc.index,   fc.values,   color=C[1], lw=2, linestyle="--", label="ETS forecast")
ax.fill_between(fc.index, pi_80_lo, pi_80_hi, color=C[1], alpha=0.25, label="80% PI")
ax.fill_between(fc.index, pi_95_lo, pi_95_hi, color=C[1], alpha=0.12, label="95% PI")
ax.axvline(test.index[0], color="grey", lw=1.5, linestyle=":", label="Forecast origin")
ax.set(xlabel="Date", ylabel="Passengers",
       title="ETS forecast with bootstrap prediction intervals
(wider = more uncertain)")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x,_: f"{int(x):,}"))
plt.tight_layout()
plt.show()

coverage_80 = np.mean((test.values >= pi_80_lo) & (test.values <= pi_80_hi))
coverage_95 = np.mean((test.values >= pi_95_lo) & (test.values <= pi_95_hi))
print(f"80% PI coverage: {coverage_80:.0%}  (should be ~80%)")
print(f"95% PI coverage: {coverage_95:.0%}  (should be ~95%)")
print(f"Forecast MAPE:   {mape(test.values, fc.values):.1f}%")